# MLP vs Sequence Models: When MLP Works and When It Fails
This notebook shows different use cases and explains why MLPs can fail on sequence data.
We build an end-to-end example comparing an MLP vs an RNN on a simple time-series task.

## 1) Quick intuition
- **MLP**: treats input as a fixed-size vector. No notion of order or time.
- **RNN/GRU/LSTM**: processes sequences step-by-step, capturing order and temporal dependencies.


## 2) Use cases and model fit
**MLP is good for:**
- Tabular data (e.g., customer features, medical records)
- Fixed-size feature vectors

**MLP can fail for:**
- Time series (order matters)
- Text (word order matters)
- Audio/sensor streams (temporal patterns)

**Why?**
If you flatten a sequence into a vector, an MLP can’t easily learn temporal structure.


## 3) End-to-end example: next-step prediction
We create a toy signal and try to predict the next value from a window of past values.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)

In [ ]:
# Generate a smooth time series
t = torch.linspace(0, 60, steps=500)
signal = torch.sin(t * 0.2) + 0.1 * torch.randn_like(t)

# Build sequences
seq_len = 20
X = []
y = []
for i in range(len(signal) - seq_len):
    X.append(signal[i:i+seq_len])
    y.append(signal[i+seq_len])
X = torch.stack(X)              # (N, seq_len)
y = torch.stack(y).unsqueeze(1) # (N, 1)

# Train/test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('X_train:', X_train.shape, 'y_train:', y_train.shape)

### 3.1 Baseline MLP (sequence flattened)
Here we treat the sequence as a flat vector. This ignores order beyond position index.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

mlp = MLP(input_dim=seq_len, hidden_dim=64)
criterion = nn.MSELoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.01)

for epoch in range(80):
    mlp.train()
    optimizer.zero_grad()
    preds = mlp(X_train)
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'MLP epoch {epoch+1}: loss={loss.item():.4f}')

mlp.eval()
with torch.no_grad():
    test_loss = criterion(mlp(X_test), y_test)
print('MLP test loss:', test_loss.item())

### 3.2 RNN model (sequence-aware)
RNN processes input step-by-step and can capture temporal patterns.

In [ ]:
class RNNRegressor(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

# Reshape for RNN: (batch, seq_len, features)
X_train_rnn = X_train.unsqueeze(-1)
X_test_rnn = X_test.unsqueeze(-1)

rnn = RNNRegressor(input_size=1, hidden_size=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(rnn.parameters(), lr=0.01)

for epoch in range(80):
    rnn.train()
    optimizer.zero_grad()
    preds = rnn(X_train_rnn)
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'RNN epoch {epoch+1}: loss={loss.item():.4f}')

rnn.eval()
with torch.no_grad():
    test_loss = criterion(rnn(X_test_rnn), y_test)
print('RNN test loss:', test_loss.item())

## 4) Why MLP can fail (summary)
- MLP treats each position independently and lacks temporal recurrence.
- It can memorize short patterns, but struggles with longer dependencies.
- Sequence models maintain hidden state to capture order and context.

## 5) When to use what
- **MLP**: tabular data, fixed-size features, embeddings.
- **RNN/GRU/LSTM**: sequence data with ordering (time series, text, signals).
- **CNN/Transformers**: spatial data (images) or long-range sequences.

## 6) Assignments
1) Replace RNN with GRU and compare test loss.
2) Increase `seq_len` to 50 and compare MLP vs RNN performance.
3) Add noise to the signal and see which model is more robust.
4) Try a 1D CNN and compare results.